# Modeling

In [1]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.metrics import root_mean_squared_log_error
from numpy import expm1
from pathlib import Path

In [2]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
MODEL_DATA_DIR = DATA_DIR / "data_model"

In [3]:
df_train = pd.read_csv(MODEL_DATA_DIR / "train_model.csv")

In [4]:
X_train = df_train.drop(columns=['SALEPRICE_LOG'])
y_train = df_train['SALEPRICE_LOG']

In [5]:
df_train.head()

,ID,MSSUBCLASS,MSZONING,HASREGULARLOTSHAPE,NEIGHBORHOOD,HOUSESTYLE,OVERALLQUAL,YEARREMODADD,EXTERIOR1ST,EXTERQUAL,...,HASWOODDECKSF,WOODDECKSF_LOG,HASMASVNRAREA,MASVNRAREA_LOG,HAS3SSNPORCH,HASLOWQUALFINSF,HASSCREENPORCH,HASENCLOSEDPORCH,HASLOTFRONTAGE,SALEPRICE_LOG
0,1,12.340706,12.1001,True,12.164080,12.212548,2,5,12.219079,3,...,False,0.000000,True,5.283204,False,False,False,False,True,12.247699
1,2,12.067974,12.1001,True,12.074029,12.011076,2,31,11.878755,2,...,True,5.700444,False,0.000000,False,False,False,False,True,12.109016
2,3,12.340706,12.1001,False,12.164080,12.212548,2,6,12.219079,3,...,False,0.000000,True,5.093750,False,False,False,False,True,12.317171
3,4,11.972299,12.1001,False,12.224805,12.212548,2,36,11.844305,2,...,False,0.000000,False,0.000000,False,False,False,True,True,11.849405
4,5,12.340706,12.1001,False,12.659042,12.212548,3,8,12.219079,3,...,True,5.262690,True,5.860786,False,False,False,False,True,12.429220


In [6]:
pca_analysis = PCA(random_state=0) 
pca_analysis.fit(X_train)

individual_variance = pca_analysis.explained_variance_ratio_
accumulated_variance = individual_variance.cumsum()

pca_table = pd.DataFrame({
    'COMPONENT': range(1, len(accumulated_variance) + 1),
    'INDIVIDUAL_VARIANCE': (individual_variance).round(4),
    'ACCUMULATED_VARIANCE': (accumulated_variance).round(4),
    'DELTA': (pd.Series(accumulated_variance).diff().fillna(accumulated_variance[0]) * 100).round(4)
})

pca_table.head()

,COMPONENT,INDIVIDUAL_VARIANCE,ACCUMULATED_VARIANCE,DELTA
0,1,0.9975,0.9975,99.7513
1,2,0.0024,0.9999,0.2356
2,3,0.0000,0.9999,0.0047
3,4,0.0000,0.9999,0.0033
4,5,0.0000,1.0000,0.0016


In [7]:
models = {
    'Ridge':             Ridge(random_state=0),
    'Lasso':             Lasso(random_state=0),
    'ElasticNet':        ElasticNet(random_state=0),
    'Bayesian Ridge':    BayesianRidge(),
    # 'Huber':             HuberRegressor(max_iter=1000),
    'KNN Regressor':     KNeighborsRegressor(),
    'SVR':               SVR(),
    'Random Forest':     RandomForestRegressor(random_state=0),
    'Extra Trees':       ExtraTreesRegressor(random_state=0),
    'Gradient Boosting': GradientBoostingRegressor(random_state=0),
    'XGBoost:': xgb.XGBRegressor(random_state=0)
}

kfold = model_selection.KFold(n_splits=5, shuffle=True, random_state=0)

In [8]:
results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('model', model)
    ])
    scores = model_selection.cross_validate(
        pipeline,
        X_train,
        y_train,
        scoring='neg_root_mean_squared_log_error',
        cv=kfold,
        n_jobs=-1
    )
    results.append({
        'MODEL': name.upper(),
        'RMSLE_MEAN': -scores['test_score'].mean(),
        'RMSLE_STD':   scores['test_score'].std()
    })

In [9]:
df_exploration = pd.DataFrame(results)
df_exploration.columns = df_exploration.columns.str.upper()
df_exploration = df_exploration.round(4)

df_exploration = df_exploration.sort_values(by='RMSLE_MEAN').reset_index(drop=True)

df_exploration

,MODEL,RMSLE_MEAN,RMSLE_STD
0,RIDGE,0.0111,0.0006
1,BAYESIAN RIDGE,0.0111,0.0006
2,GRADIENT BOOSTING,0.0111,0.0006
3,RANDOM FOREST,0.0115,0.0010
4,XGBOOST:,0.0117,0.0009
5,EXTRA TREES,0.0120,0.0011
6,ELASTICNET,0.0246,0.0009
7,SVR,0.0247,0.0010
8,LASSO,0.0248,0.0010
9,KNN REGRESSOR,0.0252,0.0007


## GRADIENT BOOSTING

In [10]:
df_exploration_analysis = df_exploration.copy()
df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']] = df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']].rank(0, numeric_only=True, method='min', ascending=True)
df_exploration_analysis['POINTS'] = df_exploration_analysis['RMSLE_MEAN'] + df_exploration_analysis['RMSLE_STD'] 

df_exploration_analysis = df_exploration_analysis.sort_values(by='POINTS', ascending=True).reset_index(drop=True)
df_exploration_analysis

,MODEL,RMSLE_MEAN,RMSLE_STD,POINTS
0,RIDGE,1.0,1.0,2.0
1,BAYESIAN RIDGE,1.0,1.0,2.0
2,GRADIENT BOOSTING,1.0,1.0,2.0
3,XGBOOST:,5.0,5.0,10.0
4,RANDOM FOREST,4.0,7.0,11.0
5,ELASTICNET,7.0,5.0,12.0
6,KNN REGRESSOR,10.0,4.0,14.0
7,SVR,8.0,7.0,15.0
8,EXTRA TREES,6.0,10.0,16.0
9,LASSO,9.0,7.0,16.0


In [11]:
gb_model = GradientBoostingRegressor(random_state=0)

param_distributions = {
        "n_estimators": [500, 525, 550],
        "learning_rate": [0.045, 0.05, 0.06],
        "max_depth": [6, 7], # experimentacao
        "min_samples_split": [90, 100, 110], #dobro do min_sample_leaf
        "min_samples_leaf": [45, 50, 55], # Geralmente, na faixa de 1-5%
        "max_features": [0.8, 0.85, 0.90], # um valor classico é \sqrt (no caso, 4 = 0.25)
        "subsample": [0.80, 0.85, 0.90] # depende do tamanho do modelo, cria generalização
    }

random_search = RandomizedSearchCV(
        estimator=gb_model,
        param_distributions=param_distributions,
        n_jobs=-1,
        random_state=0,
        scoring='neg_root_mean_squared_log_error'
    )

random_search.fit(X_train, y_train)

model_best_gb = random_search.best_estimator_
model_best_gb_score = random_search.best_score_

print(model_best_gb_score)
print(model_best_gb)

-0.010960602381113285
GradientBoostingRegressor(learning_rate=0.045, max_depth=7, max_features=0.8,
                          min_samples_leaf=50, min_samples_split=90,
                          n_estimators=550, random_state=0, subsample=0.85)


In [12]:
param_grid_refined = {
    'learning_rate':     [0.05],
    'max_depth':         [4, 5, 6],     
    'max_features':      [0.7, 0.8, 1.0],
    'min_samples_leaf':  [45, 50, 55], 
    'min_samples_split': [140, 160, 180], 
    'n_estimators':      [225, 275, 325],
    'subsample':         [0.45, 0.55],
}

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid_refined,
    n_jobs=-1,
    scoring='neg_root_mean_squared_log_error',
    return_train_score=True
)

grid_search.fit(X_train, y_train)

model_best_gb = grid_search.best_estimator_
model_best_gb_score = grid_search.best_score_
y_pred = model_best_gb.predict(X_train)

print(round(root_mean_squared_log_error(y_true=y_train, y_pred=y_pred), 4))
print(model_best_gb_score)
print(model_best_gb)

KeyboardInterrupt: 

In [ ]:
# ANALISE HERE
gb_model_gap = grid_search.cv_results_['mean_train_score'][grid_search.best_index_] - grid_search.best_score_

print("GAP: ",gb_model_gap )

GAP:  0.0027104904494643246
